In [1]:
# ======================================================
# INSTALL DEPENDENCIES
# ======================================================

!pip install -q ultralytics pyyaml

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics

ultralytics.checks()

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.4/107.7 GB disk)


In [2]:
# ======================================================
# EXTRACT DATASET
# ======================================================

import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/Project5_Ag_Crop and weed detection.zip"

EXTRACT_PATH = "/content/agri_dataset"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully!")

for root, dirs, files in os.walk(EXTRACT_PATH):
    print(root)
    break

Dataset extracted successfully!
/content/agri_dataset


In [37]:
# ======================================================
# DIAGNOSE DATASET STRUCTURE
# ======================================================

import os

print("Listing contents of /content/agri_dataset recursively:")
for root, dirs, files in os.walk("/content/agri_dataset"):
    level = root.replace("/content/agri_dataset", ".").count(os.sep)
    indent = '    ' * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '    ' * (level + 1)
    for d in dirs:
        print(f'{subindent}{d}/')
    for f in files:
        print(f'{subindent}{f}')


Listing contents of /content/agri_dataset recursively:
agri_dataset/
    Project5_Ag_Crop and weed detection/
    Project5_Ag_Crop and weed detection/
        Project5_Ag_Crop and weed detection.docx
        Project5_Ag_Crop and weed detection.zip
        ~$oject1.docx


In [39]:
# ======================================================
# EXTRACT NESTED DATASET ZIP
# ======================================================

import zipfile
import os

NESTED_ZIP_PATH = "/content/agri_dataset/Project5_Ag_Crop and weed detection/Project5_Ag_Crop and weed detection.zip"

FINAL_EXTRACT_PATH = "/content/agri_dataset/final_data"

os.makedirs(FINAL_EXTRACT_PATH, exist_ok=True)

print(f"Extracting nested zip file: {NESTED_ZIP_PATH} to {FINAL_EXTRACT_PATH}")
with zipfile.ZipFile(NESTED_ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(FINAL_EXTRACT_PATH)

print("Nested dataset extracted successfully! Now listing its contents:")
for root, dirs, files in os.walk(FINAL_EXTRACT_PATH):
    level = root.replace(FINAL_EXTRACT_PATH, ".").count(os.sep)
    indent = '    ' * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '    ' * (level + 1)
    for d in dirs:
        print(f'{subindent}{d}/')
    for f in files:
        print(f'{subindent}{f}')


Extracting nested zip file: /content/agri_dataset/Project5_Ag_Crop and weed detection/Project5_Ag_Crop and weed detection.zip to /content/agri_dataset/final_data
Nested dataset extracted successfully! Now listing its contents:
final_data/
    agri_data/
    classes.txt
    agri_data/
        data/
        data/
            agri_0_5773.txt
            agri_0_7925.txt
            agri_0_3088.jpeg
            agri_0_5661.jpeg
            agri_0_7716.txt
            agri_0_9489.jpeg
            agri_0_7943.txt
            agri_0_7104.jpeg
            agri_0_8794.txt
            agri_0_2040.txt
            agri_0_8623.jpeg
            agri_0_8838.jpeg
            agri_0_1572.jpeg
            agri_0_550.jpeg
            agri_0_2513.jpeg
            agri_0_8246.txt
            agri_0_4937.txt
            agri_0_864.txt
            agri_0_1707.jpeg
            agri_0_4951.txt
            agri_0_6292.txt
            agri_0_5858.txt
            agri_0_7579.jpeg
            agri_0_5744.jpeg
     

In [42]:
# ======================================================
# CREATE TRAIN / VAL SPLIT
# ======================================================

import os
import random
import shutil

RAW_DATA_DIR = "/content/agri_dataset/final_data/agri_data/data"

OUTPUT_DIR = "/content/data"

TRAIN_RATIO = 0.8

for folder in [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]:
    os.makedirs(os.path.join(OUTPUT_DIR, folder), exist_ok=True)

# List all image files directly from RAW_DATA_DIR
all_files = os.listdir(RAW_DATA_DIR)
images = [f for f in all_files if f.endswith(('.jpg', '.jpeg', '.png'))]

random.seed(42)
random.shuffle(images)

split_idx = int(len(images) * TRAIN_RATIO)

train_images = images[:split_idx]
val_images = images[split_idx:]

def copy_dataset(files, subset):

    copied = 0

    for img in files:

        label = os.path.splitext(img)[0] + ".txt"

        # Construct paths directly from RAW_DATA_DIR
        src_img = os.path.join(RAW_DATA_DIR, img)
        src_lbl = os.path.join(RAW_DATA_DIR, label)

        if os.path.exists(src_lbl):

            shutil.copy(
                src_img,
                os.path.join(
                    OUTPUT_DIR,
                    "images",
                    subset,
                    img
                )
            )

            shutil.copy(
                src_lbl,
                os.path.join(
                    OUTPUT_DIR,
                    "labels",
                    subset,
                    label
                )
            )

            copied += 1

    print(f"{subset}: {copied} samples copied")

copy_dataset(train_images, "train")
copy_dataset(val_images, "val")

train: 1040 samples copied
val: 260 samples copied


In [30]:
# ======================================================
# CREATE YOLO CONFIG
# ======================================================

import yaml

config = {
    "path": "/content/data",
    "train": "images/train",
    "val": "images/val",
    "nc": 2,
    "names": ["crop", "weed"]
}

with open("/content/dataset.yaml", "w") as f:
    yaml.dump(config, f)

In [ ]:
# ======================================================
# TRAIN YOLOv8
# ======================================================

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/dataset.yaml",
    epochs=50,
    imgsz=512,
    batch=16,
    device="cpu",
    project="/content/drive/MyDrive/Crop_Weed_Project",
    name="crop_weed_detection"
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=crop_weed_detection-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pe

In [ ]:
# ======================================================
# TEST MODEL
# ======================================================

import os
import random
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

MODEL_PATH = "/content/drive/MyDrive/Crop_Weed_Project/crop_weed_detection/weights/best.pt"

model = YOLO(MODEL_PATH)

VAL_DIR = "/content/data/images/val"

images = [
    os.path.join(VAL_DIR, x)
    for x in os.listdir(VAL_DIR)
]

sample_images = random.sample(images, 3)

for img in sample_images:

    result = model.predict(
        img,
        conf=0.25
    )

    annotated = result[0].plot()

    print(os.path.basename(img))

    cv2_imshow(annotated)